<a href="https://colab.research.google.com/github/samuel-1-avson/RGT-NSS/blob/main/Prompt_Engineering_and_Safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="setup"></a>
## 1. Setup & Introduction

In [1]:
# Install required libraries
!pip install transformers torch openai --quiet

print("✅ Installation complete!")

✅ Installation complete!


In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)
import json
import re
from typing import List, Dict, Any
import matplotlib.pyplot as plt
import numpy as np

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Using device: {device}")

# Load a small model for demonstrations
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Loaded {model_name}")

def generate(prompt, max_length=100, temperature=0.7, **kwargs):
    """Helper function for text generation."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=len(inputs['input_ids'][0]) + max_length,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )

    generated = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    return generated.strip()

🔥 Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Loaded gpt2


<a name="patterns"></a>
## 2. The 5 Prompting Patterns

These patterns form the foundation of effective prompt engineering.

### Pattern 1: Zero-Shot Prompting

Direct instruction without examples. Best for simple, well-defined tasks.

In [3]:
# Zero-shot examples
zero_shot_prompts = [
    {
        'task': 'Sentiment Classification',
        'prompt': '''Classify the sentiment of this review as positive, negative, or neutral:
"The product arrived on time but the packaging was damaged."

Sentiment:''',
    },
    {
        'task': 'Text Summarization',
        'prompt': '''Summarize the following in one sentence:
Artificial intelligence has transformed numerous industries, from healthcare to finance.
Machine learning algorithms can now detect diseases earlier than human doctors.

Summary:''',
    },
    {
        'task': 'Code Explanation',
        'prompt': '''Explain what this Python function does:
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

Explanation:''',
    },
]

print("🔹 Pattern 1: Zero-Shot Prompting\n")
print("=" * 70)

for example in zero_shot_prompts:
    print(f"\n📋 Task: {example['task']}")
    print("-" * 50)
    print(f"Prompt: {example['prompt'][:100]}...")
    print("\nOutput:")
    result = generate(example['prompt'], max_length=50, temperature=0.3)
    print(f"  {result}")

🔹 Pattern 1: Zero-Shot Prompting


📋 Task: Sentiment Classification
--------------------------------------------------
Prompt: Classify the sentiment of this review as positive, negative, or neutral:
"The product arrived on tim...

Output:
  "I'm not sure what to make of this. I'm not sure what to make of this. I'm not sure what to make of this. I'm not sure what to make of this. I'm not sure what to make of this

📋 Task: Text Summarization
--------------------------------------------------
Prompt: Summarize the following in one sentence:
Artificial intelligence has transformed numerous industries...

Output:
  Artificial intelligence has transformed many industries, from healthcare to finance.

The next generation of artificial intelligence is already in the process of being developed.

But there is still a long way to go.

The next generation of artificial intelligence is

📋 Task: Code Explanation
--------------------------------------------------
Prompt: Explain what this Python fun

### Pattern 2: Few-Shot Prompting

Provide examples to establish the desired pattern.

In [4]:
# Few-shot examples
few_shot_prompts = [
    {
        'task': 'Entity Extraction',
        'prompt': '''Extract the person's name and age from the text:

Text: "John Smith celebrated his 45th birthday yesterday."
Name: John Smith
Age: 45

Text: "Maria Garcia just turned 30 and got promoted."
Name: Maria Garcia
Age: 30

Text: "Robert Chen is celebrating 52 years today."
Name:''',
    },
    {
        'task': 'Format Conversion',
        'prompt': '''Convert the date to YYYY-MM-DD format:

Input: "March 5th, 2024"
Output: 2024-03-05

Input: "December 25, 2023"
Output: 2023-12-25

Input: "January 1, 2025"
Output:''',
    },
]

print("\n🔹 Pattern 2: Few-Shot Prompting\n")
print("=" * 70)

for example in few_shot_prompts:
    print(f"\n📋 Task: {example['task']}")
    print("-" * 50)
    print(f"Prompt: {example['prompt'][:150]}...")
    print("\nOutput:")
    result = generate(example['prompt'], max_length=30, temperature=0.3)
    print(f"  {result}")


🔹 Pattern 2: Few-Shot Prompting


📋 Task: Entity Extraction
--------------------------------------------------
Prompt: Extract the person's name and age from the text:

Text: "John Smith celebrated his 45th birthday yesterday."
Name: John Smith
Age: 45

Text: "Maria Ga...

Output:
  Robert Chen

Age: 52

Text: "John Smith's birthday is July 28th. He's been married for over 50 years."

📋 Task: Format Conversion
--------------------------------------------------
Prompt: Convert the date to YYYY-MM-DD format:

Input: "March 5th, 2024"
Output: 2024-03-05

Input: "December 25, 2023"
Output: 2023-12-25

Input: "January 1,...

Output:
  2025-03-01

Input: "March 5th, 2025"

Output: 2025-03-01

Input: "


### Pattern 3: Chain-of-Thought (CoT)

Encourage step-by-step reasoning for complex problems.

In [5]:
# Chain-of-thought comparison
math_problem = "A store has 25 apples. They sell 8 in the morning and receive a delivery of 15 more in the afternoon. How many apples do they have?"

without_cot = f"Q: {math_problem}\nA:"
with_cot = f"""Q: A farmer has 10 sheep. 3 die and he buys 5 more. How many sheep does he have?
A: Let's think step by step.
   - Start with 10 sheep
   - 3 die: 10 - 3 = 7 sheep
   - Buy 5 more: 7 + 5 = 12 sheep
   - Answer: 12

Q: {math_problem}
A: Let's think step by step."""

print("\n🔹 Pattern 3: Chain-of-Thought Prompting\n")
print("=" * 70)

print("\n❌ Without CoT:")
print("-" * 50)
print(f"Prompt: {without_cot}")
result = generate(without_cot, max_length=50, temperature=0.3)
print(f"Output: {result}")

print("\n✅ With CoT:")
print("-" * 50)
print(f"Prompt: {with_cot[:200]}...")
result = generate(with_cot, max_length=100, temperature=0.3)
print(f"Output: {result}")


🔹 Pattern 3: Chain-of-Thought Prompting


❌ Without CoT:
--------------------------------------------------
Prompt: Q: A store has 25 apples. They sell 8 in the morning and receive a delivery of 15 more in the afternoon. How many apples do they have?
A:
Output: They have about 10 apples.
Q: How many apples do they have?
A: They have about 5 apples.
Q: How many apples do they have?
A: They have about 5 apples.
Q: How many apples

✅ With CoT:
--------------------------------------------------
Prompt: Q: A farmer has 10 sheep. 3 die and he buys 5 more. How many sheep does he have?
A: Let's think step by step.
   - Start with 10 sheep
   - 3 die: 10 - 3 = 7 sheep
   - Buy 5 more: 7 + 5 = 12 sheep
  ...
Output: - Start with 25 apples

Q: A store has 25 apples. They sell 8 in the morning and receive a delivery of 15 more in the afternoon. How many apples do they have?

A: Let's think step by step.

Q: A store has 25 apples. They sell 8 in the morning and receive a delivery of 15 more in the

### Pattern 4: Role-Based Prompting

Assign a specific role to activate relevant knowledge.

In [6]:
# Role-based examples
roles = [
    {
        'role': 'Python Mentor',
        'prompt': '''You are an experienced Python mentor teaching a beginner.
Explain what a list comprehension is, provide a simple example, and suggest
a practice exercise. Keep your explanation friendly and avoid jargon.''',
    },
    {
        'role': 'Security Analyst',
        'prompt': '''You are a cybersecurity analyst reviewing code for vulnerabilities.
Review this function and identify any security issues:

def login(username, password):
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    return execute_query(query)''',
    },
    {
        'role': 'Creative Writer',
        'prompt': '''You are a creative writing coach helping an aspiring author.
Provide feedback on this opening paragraph and suggest improvements:

"It was a dark and stormy night. The rain fell heavily. John was scared."''',
    },
]

print("\n🔹 Pattern 4: Role-Based Prompting\n")
print("=" * 70)

for example in roles:
    print(f"\n👤 Role: {example['role']}")
    print("-" * 50)
    print(f"Prompt: {example['prompt'][:100]}...")
    print("\nOutput:")
    result = generate(example['prompt'], max_length=150, temperature=0.7)
    print(f"  {result[:200]}...")


🔹 Pattern 4: Role-Based Prompting


👤 Role: Python Mentor
--------------------------------------------------
Prompt: You are an experienced Python mentor teaching a beginner. 
Explain what a list comprehension is, pro...

Output:
  This is best if you are writing a Python script. You can use Python's built-in support for list comprehension.
If you are writing a Python script, use the following syntax:
>>> a = list ( 'a' ) >>> a ...

👤 Role: Security Analyst
--------------------------------------------------
Prompt: You are a cybersecurity analyst reviewing code for vulnerabilities. 
Review this function and identi...

Output:
  Now you can check your code to see what's going on.

Conclusion

Do you have any comments on this tutorial? Want to share it with others?

Want to help out?

Share this: Tweet


Like this: Like Loadin...

👤 Role: Creative Writer
--------------------------------------------------
Prompt: You are a creative writing coach helping an aspiring author. 
Provide feedb

### Pattern 5: Function Calling

Generate structured outputs for API integration.

In [7]:
# Function calling examples
function_prompts = [
    {
        'name': 'Weather API',
        'prompt': '''You have access to the following function:

function get_weather(location: string, unit: 'celsius' | 'fahrenheit')

User: "What's the weather like in Tokyo?"

Response format:
{"function": "get_weather", "arguments": {"location": "Tokyo", "unit": "celsius"}}

---

User: "Will it be hot in Phoenix tomorrow?"

Response format:
{"function":''',
    },
    {
        'name': 'Calculator',
        'prompt': '''You have access to the following functions:

function add(a: number, b: number)
function subtract(a: number, b: number)
function multiply(a: number, b: number)
function divide(a: number, b: number)

User: "What is 45 plus 67?"

Response format:
{"function": "add", "arguments": {"a": 45, "b": 67}}

---

User: "Calculate 100 divided by 4"

Response format:
{"function":''',
    },
]

print("\n🔹 Pattern 5: Function Calling\n")
print("=" * 70)

for example in function_prompts:
    print(f"\n⚙️ {example['name']}")
    print("-" * 50)
    print(f"Prompt: {example['prompt'][:150]}...")
    print("\nOutput:")
    result = generate(example['prompt'], max_length=100, temperature=0.3)
    print(f"  {result}")


🔹 Pattern 5: Function Calling


⚙️ Weather API
--------------------------------------------------
Prompt: You have access to the following function:

function get_weather(location: string, unit: 'celsius' | 'fahrenheit')

User: "What's the weather like in ...

Output:
  "get_weather", "arguments": {"location": "Phoenix", "unit": "celsius"}}

---

User: "Will it be sunny in Phoenix tomorrow?"

Response format:

{"function": "get_weather", "arguments": {"location": "Phoenix", "unit": "celsius"}}

---

User: "Will it be raining in Phoenix tomorrow?"

Response format:

{"function":

⚙️ Calculator
--------------------------------------------------
Prompt: You have access to the following functions:

function add(a: number, b: number)
function subtract(a: number, b: number)
function multiply(a: number, b...

Output:
  "calculate", "arguments": {"a": 100, "b": 4}}

---

User: "Calculate 100 divided by 4"

Response format:

{"function": "calculate", "arguments": {"a": 100, "b": 4}}

---

User

<a name="testing"></a>
## 3. Testing & Iteration

How to systematically test and improve your prompts.

In [8]:
# Prompt testing framework
class PromptTester:
    """Framework for testing prompt variations."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.results = []

    def test_prompt(self, name: str, prompt: str, test_cases: List[str], **gen_kwargs):
        """Test a prompt against multiple test cases."""
        print(f"\n📋 Testing: {name}")
        print("=" * 60)

        case_results = []
        for i, test_case in enumerate(test_cases, 1):
            full_prompt = prompt.format(input=test_case)
            output = self._generate(full_prompt, **gen_kwargs)
            case_results.append({
                'input': test_case,
                'output': output
            })
            print(f"\n  Test {i}:")
            print(f"    Input:  {test_case}")
            print(f"    Output: {output}")

        self.results.append({
            'name': name,
            'prompt': prompt,
            'results': case_results
        })

    def _generate(self, prompt, max_length=50, temperature=0.3):
        """Generate text."""
        inputs = self.tokenizer(prompt, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=len(inputs['input_ids'][0]) + max_length,
                temperature=temperature,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True).strip()

    def compare(self):
        """Compare all tested prompts."""
        print("\n" + "=" * 70)
        print("📊 COMPARISON SUMMARY")
        print("=" * 70)

        for result in self.results:
            print(f"\n🔹 {result['name']}")
            print(f"   Prompt: {result['prompt'][:50]}...")
            print(f"   Tests run: {len(result['results'])}")

# Create tester
tester = PromptTester(model, tokenizer)

# Test sentiment classification with different prompts
test_cases = [
    "This movie was absolutely fantastic!",
    "The service was terrible and slow.",
    "It was an okay experience, nothing special.",
]

prompts = {
    'Basic': 'Classify the sentiment: {input}\nSentiment:',
    'With Examples': '''Classify as positive, negative, or neutral.
Examples:
- "I love this!" → positive
- "This is bad" → negative
- "It's fine" → neutral

Now classify: {input}
Sentiment:''',
    'With Role': '''You are a sentiment analysis expert.
Analyze the sentiment of this text and respond with only: positive, negative, or neutral.

Text: {input}
Sentiment:''',
}

for name, prompt in prompts.items():
    tester.test_prompt(name, prompt, test_cases, temperature=0.3)

tester.compare()


📋 Testing: Basic

  Test 1:
    Input:  This movie was absolutely fantastic!
    Output: "I'm so happy to see this movie again!"
This movie was absolutely fantastic! Sentiment: "I'm so happy to see this movie again!" Sentiment: "I'm so happy to see this movie again!" Sentiment: "

  Test 2:
    Input:  The service was terrible and slow.
    Output: The service was terrible and slow.
Sentiment: The service was terrible and slow.
Sentiment: The service was terrible and slow.
Sentiment: The service was terrible and slow.
Sentiment: The service was terrible and slow

  Test 3:
    Input:  It was an okay experience, nothing special.
    Output: I'm glad you enjoyed it.
Sentiment: I'm glad you liked it.
Sentiment: I'm glad you liked it.
Sentiment: I'm glad you liked it.
Sentiment: I'm glad you liked it

📋 Testing: With Examples

  Test 1:
    Input:  This movie was absolutely fantastic!
    Output: - "I love this movie" → positive

- "I'm so glad you liked it" → negative

- "I'm so happy yo

<a name="failures"></a>
## 4. Failure Cases

Understanding common failure modes helps you design better prompts.

In [9]:
# Demonstrate failure cases
failure_examples = {
    'Hallucination': {
        'prompt': 'Who won the Nobel Prize in Physics in 2050?',
        'issue': 'Model may generate plausible-sounding but false information',
        'mitigation': 'Ask model to express uncertainty or verify facts',
    },
    'Instruction Override': {
        'prompt': '''Classify this as positive or negative (only respond with one word):
This movie was great!
Sentiment:''',
        'issue': 'Model may ignore constraints and give verbose response',
        'mitigation': 'Repeat critical constraints, use explicit formatting',
    },
    'Inconsistent Formatting': {
        'prompt': '''Extract name, age, city in JSON format:
John Smith is 45 and lives in New York.
JSON:''',
        'issue': 'Output format may vary between calls',
        'mitigation': 'Provide format examples, use post-processing',
    },
}

print("⚠️ Common Failure Cases\n")
print("=" * 70)

for failure_type, example in failure_examples.items():
    print(f"\n🔴 {failure_type}")
    print("-" * 50)
    print(f"Prompt: {example['prompt']}")
    print(f"\nOutput:")
    result = generate(example['prompt'], max_length=100, temperature=0.7)
    print(f"  {result}")
    print(f"\n⚠️ Issue: {example['issue']}")
    print(f"✅ Mitigation: {example['mitigation']}")

⚠️ Common Failure Cases


🔴 Hallucination
--------------------------------------------------
Prompt: Who won the Nobel Prize in Physics in 2050?

Output:
  Rosenberg's work has been criticised by many critics. Some argue that his work is inherently unethical and that he should be punished for it. Others argue that he has simply ignored the scientific evidence and that he has failed to follow the rules set out in an international treaty.

But he has done exactly what has been demanded by the Nobel Committee.

Rosenberg became a member of the committee in 2007, following its mandate to select the new president of the Organisation for Economic

⚠️ Issue: Model may generate plausible-sounding but false information
✅ Mitigation: Ask model to express uncertainty or verify facts

🔴 Instruction Override
--------------------------------------------------
Prompt: Classify this as positive or negative (only respond with one word):
This movie was great!
Sentiment:

Output:
  This movie was great, 

<a name="safety"></a>
## 5. Safety & Guardrails

Implementing safety measures for responsible LLM usage.

In [11]:
# Safety guardrails implementation
class SafetyGuardrails:
    """Safety guardrails for LLM applications."""

    def __init__(self):
        # Blocklist of harmful topics
        self.blocklist = [
            'harm', 'violence', 'illegal', 'weapon',
            # Add more as needed
        ]

        # Maximum input/output lengths
        self.max_input_length = 1000
        self.max_output_length = 500

    def check_input(self, text: str) -> Dict[str, Any]:
        """Check if input is safe."""
        issues = []

        # Check length
        if len(text) > self.max_input_length:
            issues.append(f"Input exceeds max length ({self.max_input_length} chars)")

        # Check blocklist
        text_lower = text.lower()
        for term in self.blocklist:
            if term in text_lower:
                issues.append(f"Input contains blocked term: '{term}'")

        return {
            'safe': len(issues) == 0,
            'issues': issues
        }

    def check_output(self, text: str) -> Dict[str, Any]:
        """Check if output is safe."""
        issues = []

        # Check for PII patterns (basic)
        pii_patterns = [
            (r'\b\d{3}-\d{2}-\d{4}\b', 'SSN'),
            (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email'),
        ]

        for pattern, pii_type in pii_patterns:
            if re.search(pattern, text):
                issues.append(f"Output may contain {pii_type}")

        return {
            'safe': len(issues) == 0,
            'issues': issues
        }

    def sanitize(self, text: str) -> str:
        """Basic sanitization of output."""
        # Remove potentially harmful patterns
        # This is a simplified example
        return text

# Demonstrate guardrails
guardrails = SafetyGuardrails()

test_inputs = [
    "What is the capital of France?",
    "This is a very long input..." * 100,  # Too long
    "How do I bake a cake?",
]

print("🛡️ Safety Guardrails Demo\n")
print("=" * 70)

for test_input in test_inputs:
    result = guardrails.check_input(test_input)
    status = "✅ Safe" if result['safe'] else "❌ Blocked"
    print(f"\n{status}: {test_input[:50]}...")
    if result['issues']:
        for issue in result['issues']:
            print(f"   ⚠️ {issue}")

🛡️ Safety Guardrails Demo


✅ Safe: What is the capital of France?...

❌ Blocked: This is a very long input...This is a very long in...
   ⚠️ Input exceeds max length (1000 chars)

✅ Safe: How do I bake a cake?...


<a name="applications"></a>
## 6. Building Safe Applications

Putting it all together into a safe LLM application.

In [12]:
class SafeLLMApplication:
    """A safe LLM application with all guardrails."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.guardrails = SafetyGuardrails()
        self.system_prompt = '''You are a helpful, harmless, and honest AI assistant.
You should:
- Provide accurate and helpful information
- Decline requests that are harmful or illegal
- Acknowledge uncertainty when you're not sure
- Keep responses concise and relevant
'''
        self.conversation_history = []

    def process(self, user_input: str, temperature: float = 0.7) -> Dict[str, Any]:
        """Process user input safely."""
        # Step 1: Input validation
        input_check = self.guardrails.check_input(user_input)
        if not input_check['safe']:
            return {
                'success': False,
                'error': 'Input validation failed',
                'issues': input_check['issues'],
                'output': None
            }

        # Step 2: Build prompt with system context
        prompt = self._build_prompt(user_input)

        # Step 3: Generate response
        try:
            output = self._generate(prompt, temperature)
        except Exception as e:
            return {
                'success': False,
                'error': f'Generation failed: {str(e)}',
                'output': None
            }

        # Step 4: Output validation
        output_check = self.guardrails.check_output(output)
        if not output_check['safe']:
            return {
                'success': False,
                'error': 'Output validation failed',
                'issues': output_check['issues'],
                'output': output
            }

        # Step 5: Update history
        self.conversation_history.append({
            'user': user_input,
            'assistant': output
        })

        return {
            'success': True,
            'output': output,
            'metadata': {
                'temperature': temperature,
                'history_length': len(self.conversation_history)
            }
        }

    def _build_prompt(self, user_input: str) -> str:
        """Build the full prompt with system context and history."""
        prompt = self.system_prompt + "\n\n"

        # Add recent conversation history
        for exchange in self.conversation_history[-3:]:
            prompt += f"User: {exchange['user']}\n"
            prompt += f"Assistant: {exchange['assistant']}\n\n"

        prompt += f"User: {user_input}\nAssistant:"
        return prompt

    def _generate(self, prompt: str, temperature: float) -> str:
        """Generate text with the model."""
        inputs = self.tokenizer(prompt, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=len(inputs['input_ids'][0]) + 100,
                temperature=temperature,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.encode('\n')[0]
            )

        generated = self.tokenizer.decode(
            outputs[0][len(inputs['input_ids'][0]):],
            skip_special_tokens=True
        )
        return generated.strip()

    def reset(self):
        """Reset conversation history."""
        self.conversation_history = []
        print("🔄 Conversation history cleared.")

# Create safe application
app = SafeLLMApplication(model, tokenizer)

# Test the application
test_queries = [
    "What is machine learning?",
    "Explain Python list comprehensions.",
]

print("🤖 Safe LLM Application Demo\n")
print("=" * 70)

for query in test_queries:
    print(f"\n👤 User: {query}")
    result = app.process(query, temperature=0.7)

    if result['success']:
        print(f"🤖 Assistant: {result['output']}")
    else:
        print(f"❌ Error: {result['error']}")
        if 'issues' in result:
            for issue in result['issues']:
                print(f"   ⚠️ {issue}")

🤖 Safe LLM Application Demo


👤 User: What is machine learning?
🤖 Assistant: Machine learning is the process of learning to understand a user's behavior or behaviour.

👤 User: Explain Python list comprehensions.
🤖 Assistant: Machine learning is the process of learning to understand a user's behavior or behaviour.
